# Thesis — 150-sentence journal profile (10 training seeds)

1. **Runtime:** Google Colab with **GPU** (recommended).
2. **Repo:** public — no GitHub token.
3. **Profile:** `--journal-paper` (Exp04 λ grid, Exp01/04, then fusion: confidence, linear SVM, RBF SVM, Gaussian NB, logistic regression, random forest). Learned routers use 5-fold OOF CV. `--journal-paper` alone would keep only linear SVM + RF; cell 8 overrides that list.
4. **Seeds:** `--num-seeds 10` (overrides journal default of 3).
5. **Data:** `data/ner_dataset_150_seed42.pkl` is **in GitHub** (no Drive upload). Clone in cell 3 delivers it.
6. **Outputs:** everything writes to **`5folds_run/`** on Drive (separate from older cross-comparison folders).
7. **Paper table:** cell **9b** builds `unified_protocol_table.xlsx` — every method re-scored on the **same** OOF outer-test tokens, so fusion is comparable to the base models under one evaluation protocol. No retraining; runs in seconds.

Docs: `thesis_overview.md` Part IV, `COLAB_README.md`.

## 1 — Mount Google Drive (recommended)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Paths (edit `WORK_ROOT` if needed)

In [ ]:
import os
from pathlib import Path

WORK_ROOT = Path("/content/drive/MyDrive/thesis_project")
# Code + tracked datasets from git on Colab disk (git on Drive often breaks).
REPO_DIR = Path("/content/thesis")
REPO_URL = "https://github.com/menachemsperka/thesis.git"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("WORK_ROOT (run outputs on Drive):", WORK_ROOT)
print("REPO_DIR (clone from GitHub):", REPO_DIR)

## 3 — Clone or update repository (public, no token)

Git checkouts on **Google Drive** often fail with `your current branch appears to be broken`. This notebook clones to **`/content/thesis`**. The 150-sentence pickle ships in the repo under **`data/`** — no manual upload.

In [ ]:
import os
import shutil
import subprocess


def _git_head_ok(repo):
    if not (repo / ".git").is_dir():
        return False
    return (
        subprocess.run(
            ["git", "-C", str(repo), "rev-parse", "HEAD"],
            capture_output=True,
        ).returncode
        == 0
    )


clone_cmd = [
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    "main",
    "--single-branch",
    REPO_URL,
    str(REPO_DIR),
]

if _git_head_ok(REPO_DIR):
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    print("Updated existing clone.")
else:
    if REPO_DIR.is_dir():
        shutil.rmtree(REPO_DIR)
    subprocess.run(clone_cmd, check=True)
    print("Cloned repository.")

corpus_pkl = REPO_DIR / "data" / "ner_dataset_150_seed42.pkl"
if not corpus_pkl.is_file():
    raise FileNotFoundError(
        f"Missing {corpus_pkl}. Ensure origin/main includes data/ner_dataset_150_seed42.pkl "
        "(re-run this cell after deleting /content/thesis)."
    )
print("Corpus from git:", corpus_pkl, "(", corpus_pkl.stat().st_size, "bytes)")

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 4 — Install dependencies

In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
# Bootstrap build tools + numerics before legacy setup.py packages (e.g. seqeval).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "seqeval", "--no-build-isolation"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Dependencies OK.")

## 5 — Colab environment (set before running experiments)

In [ ]:
import os

os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
os.environ.setdefault("THESIS_DELETE_MODELS_AFTER_TRAIN", "1")

print("THESIS_RUN_ENV =", os.environ["THESIS_RUN_ENV"])

## 6 — Check labeled dataset under `data/`

In [ ]:
from pathlib import Path

data_dir = REPO_DIR / "data"
required = "ner_dataset_150_seed42.pkl"
corpus = data_dir / required
print(f"{required}: {'OK' if corpus.is_file() else 'MISSING'}")
if not corpus.is_file():
    raise FileNotFoundError(
        f"{corpus} not found — re-run cell 3 (git clone). File is tracked on GitHub; "
        "no Drive upload needed."
    )

## 7 — Run configuration

In [ ]:
from pathlib import Path

# Isolated run root — all artifacts, splits, seeds, and checkpoints live here only.
OUTPUT_DIR = WORK_ROOT / "5folds_run"
NUM_SEEDS = 20  # 10 original + 10 extended (see cell 7a2 below — keeps the original 10 seeds' results valid)
MODELS = "dictabert,berel"
NER_DATASET_PKL = REPO_DIR / "data" / "ner_dataset_150_seed42.pkl"
# Base NER plus fusion. 06_ready is confidence fusion (no extra NER training).
# The other five are OOF routers (nested 5-fold CV). They share oof_router_cache.
EXPERIMENTS = ",".join([
    "01",
    "04",
    "06_ready",             # confidence fusion
    "06_svm_oof",           # linear SVM
    "06_svm_kernel_oof",    # RBF SVM
    "06_nb_oof",            # Gaussian naive Bayes
    "06_lr_oof",            # logistic regression
    "06_rf_oof",            # random forest
])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep False when going 10 -> 20 seeds: cell 7a2 below extends training_seeds.json
# in-place (old 10 seeds preserved + 10 new ones appended). Setting this True instead
# would regenerate ALL 20 seeds randomly, orphaning every already-completed run.
REGENERATE_TRAINING_SEEDS = False
# Set True only to skip the 9-run Exp04 λ grid (uses defaults 10/5 or cached selection).
SKIP_EXP04_LOSS_CALIBRATION = False

print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_SEEDS:", NUM_SEEDS)
print("MODELS:", MODELS)
print("EXPERIMENTS:", EXPERIMENTS)
print("NER_DATASET_PKL:", NER_DATASET_PKL)

## 7a2 — Extend training seeds 10 → 20 (run once)

Appends 10 new random seeds to the existing `training_seeds.json` **instead of**
regenerating it. The original 10 seeds (and every run already completed for them)
stay valid — only the 10 new seeds are new work. Safe to re-run: if the file
already has ≥ `NUM_SEEDS` entries, this is a no-op.

Do **not** use `--rerun-experiments` for this. That flag clears checkpoint rows
for the given *experiment ids* across **all** seeds, which would force a full
retrain of the already-completed original 10 seeds too. New seed values are, by
definition, new checkpoint rows — plain `--resume` in cell 8 picks them up
automatically without touching the first 10 seeds' results.

In [ ]:
import json
import random
from datetime import datetime
from pathlib import Path

TARGET_NUM_SEEDS = NUM_SEEDS  # 20, from cell 7
seeds_path = Path(OUTPUT_DIR) / "training_seeds.json"

if seeds_path.is_file():
    data = json.loads(seeds_path.read_text(encoding="utf-8"))
    existing = [int(s) for s in data.get("seeds", [])]
else:
    data = {}
    existing = []

if len(existing) >= TARGET_NUM_SEEDS:
    print(f"training_seeds.json already has {len(existing)} seeds (>= {TARGET_NUM_SEEDS}); nothing to add.")
else:
    need = TARGET_NUM_SEEDS - len(existing)
    rng = random.SystemRandom()
    seen = set(existing)
    new_seeds = []
    while len(new_seeds) < need:
        candidate = rng.randint(1, 2_147_483_646)
        if candidate not in seen:
            seen.add(candidate)
            new_seeds.append(candidate)
    data["seeds"] = existing + new_seeds
    data["num_seeds"] = len(data["seeds"])
    data.setdefault(
        "note",
        "Stable training seeds for THESIS_SPLIT_SEED / paired runs. "
        "Reuse this file across reruns; do not edit unless intentional.",
    )
    data["extended_at"] = datetime.now().isoformat(timespec="seconds")
    seeds_path.parent.mkdir(parents=True, exist_ok=True)
    seeds_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Extended {seeds_path}: {len(existing)} -> {len(data['seeds'])} seeds (added {len(new_seeds)} new).")
    print("Original seeds (unchanged):", existing)
    print("New seeds (added):", new_seeds)

## 7b — Preflight (GPU, corpus, seed file)

If cell 8 fails with `CalledProcessError`, **scroll up** in the output for the real `Traceback` / `[PREP]` line (the exception is only the exit code).

In [ ]:
import json
import torch
from pathlib import Path

if not NER_DATASET_PKL.is_file():
    raise FileNotFoundError(f"Missing corpus: {NER_DATASET_PKL} — run cell 3 (git pull) again.")

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARN: CPU runtime works but is very slow; use Runtime → GPU.")

seeds_path = Path(OUTPUT_DIR) / "training_seeds.json"
if seeds_path.is_file():
    stored = json.loads(seeds_path.read_text(encoding="utf-8")).get("seeds", [])
    print(f"training_seeds.json: {len(stored)} seeds on disk, NUM_SEEDS={NUM_SEEDS}")
    if len(stored) < NUM_SEEDS and not REGENERATE_TRAINING_SEEDS:
        raise ValueError(
            f"{seeds_path} has {len(stored)} seeds but NUM_SEEDS={NUM_SEEDS}. "
            "Set REGENERATE_TRAINING_SEEDS = True in cell 7 and re-run 7 → 7b → 8, "
            "or delete training_seeds.json under 5folds_run."
        )
else:
    print("training_seeds.json: will be created on first run.")

## 7c — Debug run (DISABLED for Run All — manual-only; see note below)

Run this **once, manually** if cell 8 fails with no visible error, then fix and use cell 8 for the full run.

**Commented out on purpose:** this cell unconditionally passes `--regenerate-training-seeds`,
which would overwrite `training_seeds.json` with a brand-new random 20-seed list and orphan
every already-completed run (including the 10→20 extension from cell 7a2). Safe to skip during
`Run All`; uncomment only for manual one-off debugging, then re-comment before your next full run.

In [ ]:
# NOTE: disabled for `Run All` -- this is a MANUAL debug-only cell.
# It unconditionally passes --regenerate-training-seeds, which would
# overwrite training_seeds.json with a brand-new random seed list and
# orphan every already-completed run (including the 20-seed extension
# from cell 7a2). Uncomment and run manually ONLY if cell 8 fails with
# no visible error, then re-comment before your next Run All.
# import os, sys, subprocess
#
# env = os.environ.copy()
# env.update({
#     "THESIS_RUN_ENV": "colab",
#     "WANDB_DISABLED": "true",
#     "THESIS_NER_CSV": str(NER_DATASET_PKL),
#     "THESIS_CROSS_EXPERIMENTS": EXPERIMENTS,
#     "PYTHONUNBUFFERED": "1",
# })
# dbg_argv = [
#     sys.executable, "run_cross_data_model_comparison.py",
#     "--journal-paper", "--experiments", EXPERIMENTS,
#     "--num-seeds", str(NUM_SEEDS),
#     "--regenerate-training-seeds",
#     "--resume", "--output-dir", str(OUTPUT_DIR),
#     "--exp07-source", "rerun",
#     "--models", "dictabert",
#     "--base-mode", "auto",
#     "--skip-exp04-loss-calibration",
# ]
# proc = subprocess.run(dbg_argv, cwd=str(REPO_DIR), env=env, text=True, capture_output=True)
# print("--- stdout ---\n", proc.stdout or "(empty)")
# print("--- stderr ---\n", proc.stderr or "(empty)")
# print("exit code:", proc.returncode)
print("Cell 7c (debug run) skipped intentionally -- see markdown note above.")

## 8 — Run cross-comparison (argv one row per argument)

Uses **`--journal-paper`** with **`--num-seeds 10`**. Fusion is the cell-7 `EXPERIMENTS` list (confidence + linear SVM, RBF SVM, Gaussian NB, logistic regression, random forest). `THESIS_CROSS_EXPERIMENTS` is set so `--journal-paper` does not replace that list with linear SVM + RF only.

Same command split into a Python list so each flag is its own line.

Expect a long run; use **`--resume`** on reconnect (re-run cells 1–7b, then this cell only).

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

env = os.environ.copy()
env["THESIS_RUN_ENV"] = "colab"
env["WANDB_DISABLED"] = "true"
env["THESIS_NER_CSV"] = str(NER_DATASET_PKL)
# Required: --journal-paper overwrites --experiments unless this env var is set.
env["THESIS_CROSS_EXPERIMENTS"] = EXPERIMENTS

argv = [
    sys.executable,
    "run_cross_data_model_comparison.py",
    "--journal-paper",
    "--experiments",
    EXPERIMENTS,
    "--num-seeds",
    str(NUM_SEEDS),
    "--resume",
    "--output-dir",
    str(OUTPUT_DIR),
    "--exp07-source",
    "rerun",
    "--models",
    MODELS,
    "--base-mode",
    "auto",
]
if REGENERATE_TRAINING_SEEDS:
    argv.append("--regenerate-training-seeds")
if SKIP_EXP04_LOSS_CALIBRATION:
    argv.append("--skip-exp04-loss-calibration")

print("THESIS_NER_CSV =", env["THESIS_NER_CSV"])
print("Command:")
for i, part in enumerate(argv):
    print(f"  [{i}] {part}")

env["PYTHONUNBUFFERED"] = "1"

# Stream line-by-line (guaranteed to render live in Colab, unlike plain
# subprocess.run() which can inherit the raw fd and show nothing until exit)
# and mirror everything to a log file so a dropped connection doesn't lose it.
log_path = Path(OUTPUT_DIR) / "cell8_run.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
with open(log_path, "a", encoding="utf-8") as logf, subprocess.Popen(
    argv,
    cwd=str(REPO_DIR),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

print("exit code:", proc.returncode)
if proc.returncode != 0:
    raise RuntimeError(
        f"Runner exited with code {proc.returncode}. "
        f"Read [FATAL] / Traceback lines above, or check {log_path}."
    )

## 9 — Optional: rebuild Excel/JSON only (DISABLED for Run All — no retraining needed; manual-only)

In [ ]:
# NOTE: disabled for `Run All` -- this is an OPTIONAL manual utility,
# not required after cell 8 already completed (it already exports
# everything at the end of a successful run). Uncomment only if you
# need to re-export Excel/JSON from an existing checkpoint without
# retraining anything.
# import os
# import sys
# import subprocess
#
# os.chdir(REPO_DIR)
#
# argv_rebuild = [
#     sys.executable,
#     "run_cross_data_model_comparison.py",
#     "--rebuild-from-checkpoint",
#     "--output-dir",
#     str(OUTPUT_DIR),
# ]
#
# subprocess.run(argv_rebuild, check=True)
print("Cell 9 (rebuild-only) skipped intentionally -- see markdown note above.")

## 9a — Regenerate exp07 split JSONs (run this if 9b raises `Missing .../exp07/splits`)

**Why they're missing.** `run_cross_data_model_comparison.py` only redirects the splits onto Drive inside the `--subset-sentences` branch:

```python
if subset_sentences is not None and subset_sentences > 0:
    ...
    EXP07_SPLITS_DIR = COMPARISON_DIR / "exp07" / "splits"
```

Cell 8 feeds the 150-sentence corpus through `THESIS_NER_CSV` instead of `--subset-sentences`, so that branch never ran and `EXP07_SPLITS_DIR` kept its module default of `<repo>/outputs/exp07/splits` — i.e. `/content/thesis/outputs/exp07/splits`, on the **ephemeral Colab disk**. It vanished when the training session ended.

**Nothing was lost.** The OOF result workbooks are unaffected: `THESIS_EXPERIMENTS_OUTPUT_DIR` *is* set whenever `--output-dir` is given, so they're safe on Drive under `experiment_outputs/`. Only the split JSONs went to the scratch disk.

**Why regenerating is exact, not approximate.** `regenerate_exp07_splits()` is fully deterministic and the runner calls it with no overrides, so it uses the same defaults now as it did then — `split_seed=42`, `split_ratio=0.7`, same corpus pickle, same allocator code. The regenerated JSONs are byte-identical to the originals, which is what makes the fold reconstruction in 9b valid: `sentence_id` is the position within `(train + eval)`, so the ordering has to match exactly.

No NER training — this reads the corpus and writes JSON. Seconds, CPU.

In [ ]:
import os
import sys
from pathlib import Path

os.chdir(REPO_DIR)

SPLITS_DIR = Path(OUTPUT_DIR) / "exp07" / "splits"

if (SPLITS_DIR / "split_meta.json").is_file():
    print(f"Splits already present at {SPLITS_DIR} — nothing to do.")
else:
    # Same two env vars the runner would have set; get_splits_dir() reads the second.
    os.environ["THESIS_NER_CSV"] = str(NER_DATASET_PKL)
    os.environ["THESIS_EXP07_SPLITS_DIR"] = str(SPLITS_DIR)
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)

    for sub in ("experiments", "core"):
        p = str(REPO_DIR / sub)
        if p not in sys.path:
            sys.path.insert(0, p)

    from exp07_split_artifacts import regenerate_exp07_splits

    # Defaults split_seed=42 / split_ratio=0.7 match what the runner used.
    info = regenerate_exp07_splits(clear_existing=True)
    print("dataset:          ", info.get("dataset"))
    print("source_sentences: ", info.get("source_sentences"))
    print("variants:         ", [v.get("variant") for v in info.get("variants", [])])

print("\nFiles in", SPLITS_DIR)
for p in sorted(SPLITS_DIR.glob("*.json")):
    print("  ", p.name)

## 9b — Unified single-protocol results table (**the only file you need for the paper**)

Re-scores **every** method on the **same** nested-CV outer-test tokens, so fusion becomes directly comparable to the base models. Writes `unified_protocol_table.xlsx` under `OUTPUT_DIR`.

**Two files exist, but you only open one.** `cross_comparison_latest.xlsx` is the runner's native export — cell 8 always writes it and it holds the λ grid, the loss config, and the seed-based holdout numbers. `unified_protocol_table.xlsx` is derived from it plus the OOF prediction workbooks, and this cell copies the runner's paper sheets into it verbatim. So the unified workbook is **self-contained**: results, deltas, and p-values alongside λ and the holdout reference. Write from it and ignore the other.

**Which sheet for what:** main-table cells → `fold_summary`; headline gain → `fusion_vs_base`; p-values → `paired_fold_tests`; error analysis → `token_error_summary`; Methods λ → `journal_lambda_grid`; holdout comparison → `journal_main_table`.

**Why this is needed.** Reviewers flagged that mixing 70/30 holdout numbers and OOF numbers in one table is misleading. It is worse than a presentation problem: the OOF outer folds train the base models on ~120 of 150 sentences while the holdout trains on ~105, so a mixed table credits fusion with a ~15% training-data advantage that has nothing to do with fusion. Scoring everything on the identical pooled outer-test tokens removes that confound.

**Why the existing sheets don't cover it.** `journal_paired_fold_deltas` is documented as testing every fusion method against Regular NER and Cascade NER, but `collect_oof_fold_long()` in `experiments/journal_results_export.py` keeps only experiment ids containing `_oof`, so `exp01` and `exp04` never enter `fold_long` and those contrasts are silently dropped. This cell supplies the missing per-fold base-model F1.

**No retraining.** Each OOF run already retrained Exp01/Exp04 per outer fold and stored both predictions next to gold in its `detailed_results` sheet. This is pure post-processing — runs in seconds on CPU.

**What it computes.** Direct NER (Exp01), three-step cascade (Exp04), cascade + repair (Exp05 B/I consistency re-applied), confidence fusion, and each OOF router — all as strict entity-level F1 over the same tokens, plus token-level FP/FN/type/boundary counts and paired Wilcoxon and paired *t*-tests with Holm adjustment inside each `(model, base_split_condition)` family. Grouping drops the `__seed<N>` suffix the runner appends to `split_condition`, so the 20 training seeds pool into one n=100 test per contrast rather than 20 tests of n=5.

**Prerequisite:** `build_unified_protocol_table.py` must be on `origin/main`. Commit and push it locally, then re-run **cell 3** so the clone picks it up — the preflight below fails with a clear message if it's missing.

**`--splits-dir`, not the single-JSON flags.** `sentence_id` is the position within `(train JSON + eval JSON)`, so fold membership is **condition-specific**. `--splits-dir` resolves `<split_condition>_train/eval.json` per run; a single hard-coded pair would mis-map folds for every other condition.

**Where the inputs live.** `--oof-dir` is searched **recursively**, so pointing it at `OUTPUT_DIR` picks up the per-run workbooks under `experiment_outputs/exp06_*_oof/fusion_*_oof_*.xlsx` (the runner redirects them to Drive via `THESIS_EXPERIMENTS_OUTPUT_DIR`). All seeds, models, and conditions share one directory per router, so the script de-duplicates by `(model, split_condition, seed, router)` from each file's `metrics` sheet and keeps the newest.

**Check `n_pairs` in the output.** Pairing is by `(training_seed, outer_fold)`, so 20 seeds × 5 folds should read **100**. If it reads 5, only one seed's OOF runs were found — and at *n*=5 a two-sided signed-rank test cannot go below *p*=0.0625 no matter how large the effect, so nothing will survive Holm.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)

builder = REPO_DIR / "build_unified_protocol_table.py"
if not builder.is_file():
    raise FileNotFoundError(
        f"Missing {builder}. Commit and push build_unified_protocol_table.py to origin/main, "
        "then re-run cell 3 (git pull) and this cell again."
    )

SPLITS_DIR = Path(OUTPUT_DIR) / "exp07" / "splits"
RUNNER_XLSX = Path(OUTPUT_DIR) / "cross_comparison_latest.xlsx"
UNIFIED_XLSX = Path(OUTPUT_DIR) / "unified_protocol_table.xlsx"

if not (SPLITS_DIR / "split_meta.json").is_file():
    raise FileNotFoundError(
        f"Missing {SPLITS_DIR} — run cell 9a first to regenerate the exp07 split JSONs "
        "(deterministic, no training). Cell 8 wrote them to the ephemeral /content disk, "
        "so they do not survive a session."
    )

argv_unified = [
    sys.executable,
    "build_unified_protocol_table.py",
    "--oof-dir",
    str(OUTPUT_DIR),
    # Condition-aware fold reconstruction; see the note above on why this beats --corpus-*-json.
    "--splits-dir",
    str(SPLITS_DIR),
    "--output",
    str(UNIFIED_XLSX),
]
# Copy the runner's paper sheets in verbatim (lambda grid, loss config, journal_main_table,
# the runner's own fold sheets) so unified_protocol_table.xlsx is the ONLY file you need.
if RUNNER_XLSX.is_file():
    argv_unified += ["--reference-xlsx", str(RUNNER_XLSX)]
else:
    print(
        f"NOTE: {RUNNER_XLSX.name} not found — the unified workbook will still contain the "
        "re-scored results and p-values, but not the lambda grid / loss config sheets."
    )

print("Command:")
for i, part in enumerate(argv_unified):
    print(f"  [{i}] {part}")
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Stream line-by-line like cell 8 does. Plain subprocess.run() inherits the raw fd and
# Colab then shows nothing at all until the process exits, which makes a long Drive scan
# look like a hang. Mirror to a log file so a dropped connection doesn't lose the output.
log_path = Path(OUTPUT_DIR) / "cell9b_unified.log"
with open(log_path, "a", encoding="utf-8") as logf, subprocess.Popen(
    argv_unified,
    cwd=str(REPO_DIR),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(
        f"build_unified_protocol_table.py exited with code {proc.returncode}. See {log_path}."
    )

print("\nWrote:", UNIFIED_XLSX)

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

UNIFIED_XLSX = Path(OUTPUT_DIR) / "unified_protocol_table.xlsx"
sheets = pd.read_excel(UNIFIED_XLSX, sheet_name=None)
print("Sheets:", {name: df.shape for name, df in sheets.items()})

# Main table cells: mean +/- SD over all (training_seed, outer_fold) units.
summary = sheets.get("fold_summary")
if summary is not None and not summary.empty:
    print("\n=== fold_summary (quote these cells in the main table) ===")
    print(
        summary[["model", "base_split_condition", "method", "f1_mean_pm_sd", "n_seeds", "n_folds", "n_units"]]
        .sort_values(["model", "base_split_condition", "method"])
        .to_string(index=False)
    )
else:
    print(
        "\nfold_summary is empty — the split JSONs for one or more conditions were not found, "
        "so folds could not be reconstructed. Fall back to unified_main_table (pooled F1)."
    )
    print(sheets["unified_main_table"][["model", "split_condition", "method", "f1"]].to_string(index=False))

# The headline gain, on identical tokens.
print("\n=== fusion_vs_base (abstract / results gain) ===")
print(
    sheets["fusion_vs_base"][
        ["model", "split_condition", "router", "baseline_method", "baseline_f1", "fusion_f1", "delta_f1"]
    ].to_string(index=False)
)

# Significance, Holm-adjusted within each (model, base_split_condition) family.
tests = sheets.get("paired_fold_tests")
if tests is not None and not tests.empty:
    print("\n=== paired_fold_tests (Holm-adjusted per model x condition) ===")
    print(
        tests[[
            "model", "base_split_condition", "contrast", "n_pairs", "mean_delta_f1",
            "wilcoxon_p", "wilcoxon_p_holm", "ttest_rel_p_holm", "holm_family_size", "sig_01_holm",
        ]].to_string(index=False)
    )
    # With 20 seeds x 5 folds this should read 100. If it reads 5 the seeds did not pool,
    # and no p-value can fall below 0.0625 regardless of effect size.
    print("\nn_pairs observed:", sorted(tests["n_pairs"].unique().tolist()))
else:
    print("\npaired_fold_tests is empty — needs the per-fold sheets (see fold_summary note above).")

# Matched-CV token error counts for the base models, summed over all training seeds.
errors = sheets.get("token_error_summary")
if errors is not None and not errors.empty:
    print("\n=== token_error_summary (FP / FN / type / boundary per method) ===")
    print(
        errors[[
            "model", "base_split_condition", "method", "n_seeds", "eval_tokens", "entity_tokens",
            "false_positive", "false_negative", "type_error", "boundary_error", "token_error_rate",
        ]].to_string(index=False)
    )
else:
    print("\ntoken_error_summary is empty — classify_error() could not be imported.")

## 10 — List key output files

In [ ]:
from pathlib import Path

out = Path(OUTPUT_DIR)
checks = [
    "run_manifest.json",
    "training_seeds.json",
    "data/ner_dataset_150_seed42.pkl",
    "exp07/splits/split_meta.json",
    "cross_comparison_progress_latest.json",
    "cross_comparison_latest.xlsx",
    "unified_protocol_table.xlsx",  # cell 9b — main paper table, one evaluation protocol
]
for rel in checks:
    p = out / rel
    print(f"{rel}: {'OK' if p.is_file() else 'missing'}")

for p in sorted(out.glob("consolidated_error_analysis*.xlsx"))[-2:]:
    print("workbook:", p.name)